## Diving into the details of the owa implementations.

In [82]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [83]:
import numpy as np
from quickrules.data_handling import get_dataset, test_save
from pathlib import Path
import fuzzyroughrules.fuzzy_operators as fo
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from fuzzyroughrules.rule_induction_base import RuleGenerator
from fuzzyroughrules.approximations import LowerApproximation, MulticlassMSECVXApproximation, OWALowerApproximation
from fuzzyroughrules.operators import ImplicatorInclusion, OWAImplicatorInclusion
from quickrules.data_handling import test_save
from pathlib import Path
from sklearn.metrics import balanced_accuracy_score

In [30]:
for nr in range(1, 11):
    name = 'pima' # 'bupa
    # nr = 7
    x_train, y_train, t_train = get_dataset(
        Path.cwd() / 'keel-data' / f'{name}-10-fold',
        f"{nr}tra",
        get_datatypes=True
    )
    x_test, y_test = get_dataset(
        Path.cwd() / 'keel-data' / f'{name}-10-fold',
        f"{nr}tst"
    )
    classes = list(np.unique(np.append(y_train, y_test)))
    y_train = np.array([classes.index(label) for label in y_train])
    y_test = np.array([classes.index(label) for label in y_test])
    model = RuleGenerator(
        OWAImplicatorInclusion()
    )
    model.fit(x_train, y_train, _)
    print(balanced_accuracy_score(y_test, model.predict(x_test)))

0.6562962962962963
0.7057142857142857
0.6407692307692308
0.7162962962962962
0.7062962962962962
0.7353846153846153
0.7803703703703704
0.62
0.6107407407407407
0.5837037037037037



Comparing this to acc when using a higher threshold

In [25]:
name = 'ecoli' # 'bupa
nr = 1
x_train, y_train, t_train = get_dataset(
    Path.cwd() / 'keel-data' / f'{name}-10-fold',
    f"{nr}tra",
    get_datatypes=True
)
x_test, y_test = get_dataset(
    Path.cwd() / 'keel-data' / f'{name}-10-fold',
    f"{nr}tst"
)
classes = list(np.unique(np.append(y_train, y_test)))
y_train = np.array([classes.index(label) for label in y_train])
y_test = np.array([classes.index(label) for label in y_test])
model = RuleGenerator(
    approximation=OWALowerApproximation()
)
model.fit(x_train, y_train, _)
print(balanced_accuracy_score(y_test, model.predict(x_test)))

0.14285714285714285


In [26]:
print(model.predict(x_test))

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [27]:
len(model.get_rules_as_string())

14

In [73]:
name = 'ecoli' # 'bupa
# nr = 7
x_train, y_train, t_train = get_dataset(
    Path.cwd() / 'keel-data' / f'{name}-10-fold',
    f"{nr}tra",
    get_datatypes=True
)

In [74]:
_, _, counts = np.unique(y_train, return_inverse=True, return_counts=True)

In [75]:
priors = np.divide(counts, max(counts))

In [76]:
priors

array([1.00000, 0.53488, 0.01550, 0.01550, 0.24806, 0.13953, 0.03101,
       0.36434])

In [77]:
(priors * 0.05 + 0.95) * (1-1e-6)

array([1.00000, 0.97674, 0.95077, 0.95077, 0.96240, 0.95698, 0.95155,
       0.96822])

In [81]:
if 0:
    print ( True)

Getting the new granular labels

In [86]:
X, y, t_train = get_dataset(
    Path.cwd() / 'keel-data' / f'{name}-10-fold',
    f"{nr}tra",
    get_datatypes=True
)

In [87]:
import numpy as np
from sklearn.preprocessing import normalize, MinMaxScaler
from sklearn.utils.multiclass import check_classification_targets
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
import gurobipy as gb
from fuzzyroughrules.operators import RelationTypes, triangular_relation, ImplicatorInclusion
from fuzzyroughrules.approximations import LowerApproximation, M

In [98]:
X, y = check_X_y(X, y)
check_classification_targets(y)
classes_, y, counts = np.unique(y, return_inverse=True, return_counts=True)
priors_ = np.divide(counts, max(counts))  # scaled_priors
n_classes_ = len(classes_)
n_samples_, n_features_in_ = X.shape

approximation_ = MulticlassMSECVXApproximation()
inclusion_measure_ = ImplicatorInclusion()
scaler_ = MinMaxScaler()

X = scaler_.fit_transform(X)
rel_matrix_x_ = fo.triangular_similarity(X, X)
rel_matrix_y_ = fo.discernibility_matrix(y, y)

In [110]:
n_classes_

8

In [99]:
positive_region_ = approximation_.get_approximation(X, y)

In [100]:
positive_region_

array([0.71039, 0.77494, 0.63241, 0.60968, 0.81327, 0.62731, 0.82967,
       0.82728, 0.60290, 0.79150, 0.74030, 0.77896, 0.66524, 0.78122,
       0.67817, 0.75474, 0.59215, 0.85021, 0.64293, 0.90622, 0.86948,
       0.92838, 0.67817, 0.80720, 0.72339, 0.70190, 0.77493, 0.59811,
       0.77494, 0.84736, 0.68499, 0.73031, 0.60655, 0.75645, 0.71268,
       0.72236, 0.71042, 0.73193, 0.79645, 0.62144, 0.69967, 0.55989,
       0.74448, 0.68839, 0.91524, 0.73193, 0.58139, 0.66412, 0.64392,
       0.69855, 0.78427, 0.69803, 0.73476, 0.62831, 0.72488, 0.64591,
       0.89061, 0.69424, 0.59215, 0.79459, 0.59691, 0.79830, 0.73637,
       0.85873, 0.65521, 0.76419, 0.70477, 0.75566, 0.74475, 0.65712,
       0.75876, 0.74083, 0.77493, 0.75038, 0.68825, 0.66741, 0.62440,
       0.65666, 0.63293, 0.80942, 0.64335, 0.69952, 0.78285, 0.70833,
       0.83093, 0.73802, 0.45798, 0.77598, 0.55425, 0.65357, 0.77896,
       0.70759, 0.76477, 0.71100, 0.84736, 0.80720, 0.65618, 0.66741,
       0.65252, 0.80

In [101]:
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       3, 3, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5,
       5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 7, 7, 7, 7, 7, 7, 7, 7,
       7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
       7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,

In [107]:
from fuzzyroughrules.fuzzy_operators import lukasiewicz_implicator


lukasiewicz_implicator(rel_matrix_x_, rel_matrix_y_)

array([[1.00000, 1.00000, 1.00000, ..., 0.26437, 0.36782, 0.51724],
       [1.00000, 1.00000, 1.00000, ..., 0.61364, 0.65517, 0.80460],
       [1.00000, 1.00000, 1.00000, ..., 0.08081, 0.65517, 0.80460],
       ...,
       [0.26437, 0.61364, 0.08081, ..., 1.00000, 1.00000, 1.00000],
       [0.36782, 0.65517, 0.65517, ..., 1.00000, 1.00000, 1.00000],
       [0.51724, 0.80460, 0.80460, ..., 1.00000, 1.00000, 1.00000]])

In [108]:
rel_matrix_x_

array([[1.00000, 0.52273, 0.71264, ..., 0.73563, 0.63218, 0.48276],
       [0.52273, 1.00000, 0.44318, ..., 0.38636, 0.34483, 0.19540],
       [0.71264, 0.44318, 1.00000, ..., 0.91919, 0.34483, 0.19540],
       ...,
       [0.73563, 0.38636, 0.91919, ..., 1.00000, 0.36782, 0.21839],
       [0.63218, 0.34483, 0.34483, ..., 0.36782, 1.00000, 0.82955],
       [0.48276, 0.19540, 0.19540, ..., 0.21839, 0.82955, 1.00000]])

In [109]:
rel_matrix_y_

array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 1, 1, 1],
       [0, 0, 0, ..., 1, 1, 1],
       [0, 0, 0, ..., 1, 1, 1]])

In [116]:
from cvxopt import matrix, spmatrix, solvers
from mosek import iparam
import multiprocessing

In [124]:
relation_matrix_x = rel_matrix_x_
relation_matrix_y = rel_matrix_y_

n_samples = relation_matrix_x.shape[0]
weights = np.ones(n_samples)
nn_approx = int(n_samples_)

implications = lukasiewicz_implicator(relation_matrix_x, relation_matrix_y)

q = matrix(-weights)

inter_class_relation_x = (1 - relation_matrix_y) * relation_matrix_x
tmp_important = inter_class_relation_x > 0
closest_inds = np.argpartition(-inter_class_relation_x, kth=nn_approx - 1, axis=1)[:, :nn_approx]
if_bigger_zero = np.take_along_axis(tmp_important, closest_inds, axis=1)
n_closest = np.sum(if_bigger_zero, axis=1)
tmp_conc1 = np.repeat(np.arange(n_samples), n_closest)
tmp_conc2 = closest_inds[if_bigger_zero]
matrix_inds = np.unique(
    np.sort(np.concatenate([tmp_conc1.reshape(-1, 1), tmp_conc2.reshape(-1, 1)], axis=1), axis=1), axis=0)

num_comb = matrix_inds.shape[0]
first_index_constr = np.repeat(np.arange(num_comb), 2)
first_index_bound = np.arange(num_comb, num_comb + n_samples)
second_index_constr = matrix_inds.flatten().astype(int)
second_index_bound = np.arange(n_samples)
values_constr = np.ones(2 * num_comb)
values_bound = np.ones(n_samples)
G = spmatrix(np.concatenate((values_constr, values_bound)),
             np.concatenate((first_index_constr, first_index_bound)),
             np.concatenate((second_index_constr, second_index_bound)))
right_side_constr = implications[matrix_inds[:, 0], matrix_inds[:, 1]] + 1
right_side_bound = np.ones(n_samples)
h = matrix(np.concatenate((right_side_constr, right_side_bound)))

num_threads = 1

solvers.options['mosek'] = {iparam.log: 0, iparam.num_threads: num_threads}
sol = solvers.lp(q, G, h, solver='mosek')
lambdas_opt = np.array(sol['x']).flatten()

In [122]:
np.array(sol['x'])

array([[0.86795],
       [0.85529],
       [0.77577],
       [0.71408],
       [0.94070],
       [0.70832],
       [0.92611],
       [0.93056],
       [0.68325],
       [0.98007],
       [0.87560],
       [0.89479],
       [0.81542],
       [0.90700],
       [0.75851],
       [0.84163],
       [0.67249],
       [0.93056],
       [0.73609],
       [0.99507],
       [1.00000],
       [1.00000],
       [0.75851],
       [0.88755],
       [0.80469],
       [0.81852],
       [0.85529],
       [0.69109],
       [0.85529],
       [0.95526],
       [0.77982],
       [0.82876],
       [0.73707],
       [0.86435],
       [0.79983],
       [0.84535],
       [0.79077],
       [0.81228],
       [0.87679],
       [0.73806],
       [0.78002],
       [0.64024],
       [0.87500],
       [0.77368],
       [1.00000],
       [0.81228],
       [0.66174],
       [0.76535],
       [0.70854],
       [0.77333],
       [0.87679],
       [0.78808],
       [0.81544],
       [0.72799],
       [0.84151],
       [0.